In [1]:
import sys; sys.path.insert(0, "..")
from fantasy.yahoo_client import get_my_roster, get_current_matchup, get_roster_by_team, get_league_categories
from fantasy.nba_stats import get_player_stats, get_games_this_week, batch_get_player_stats
from fantasy.analysis import project_team_categories, classify_categories
from fantasy.llm import build_matchup_prompt, ask_gemini
import pandas as pd

In [2]:
import importlib, os
from dotenv import load_dotenv
load_dotenv("/mnt/c/Users/louis/Documents/dev/.env", override=True)

import fantasy.llm
importlib.reload(fantasy.llm)
from fantasy.llm import build_matchup_prompt, ask_gemini


In [3]:
matchup = get_current_matchup()
my_roster = get_my_roster()
opp_roster = get_roster_by_team(matchup["opponent_team_key"])
categories = get_league_categories()
week_start, week_end = matchup["week_start"], matchup["week_end"]
print(f"Week {matchup['week']}: {week_start} → {week_end}")
print(f"Opponent team: {matchup['opponent_team_key']}")

[2026-03-24 11:31:36,630 DEBUG] [yahoo_oauth.oauth.__init__] Checking 
[2026-03-24 11:31:36,651 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 39077.565546274185
[2026-03-24 11:31:36,652 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN HAS EXPIRED
[2026-03-24 11:31:36,652 DEBUG] [yahoo_oauth.oauth.refresh_access_token] REFRESHING TOKEN
[2026-03-24 11:31:36,868 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 0.21536517143249512
[2026-03-24 11:31:36,869 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID
[2026-03-24 11:31:37,939 DEBUG] [yahoo_oauth.oauth.__init__] Checking 
[2026-03-24 11:31:37,944 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 1.290851354598999
[2026-03-24 11:31:37,944 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID
[2026-03-24 11:31:37,962 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 1.30930757522583
[2026-03-24 11:31:37,963 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID
[2026-03-24 11:31

Week 22: 2026-03-23 → 2026-03-29
Opponent team: 466.l.26708.t.3


In [4]:
from fantasy.nba_stats import get_week_games_detail, get_matchup_factor

def enrich_players(roster):
    with_stats = batch_get_player_stats(roster)
    enriched = []
    for p in with_stats:
        detail = get_week_games_detail(p['team_abbr'], week_start, week_end)
        mf = get_matchup_factor(p['team_abbr'], week_start, week_end)
        enriched.append({
            **p,
            'games_remaining': detail['total'],
            'games_detail': detail,
            'matchup_factor': mf,
        })
    return enriched

my_players = enrich_players(my_roster)
opp_players = enrich_players(opp_roster)


In [5]:
my_proj = project_team_categories(my_players)
opp_proj = project_team_categories(opp_players)

def to_league_cats(proj, cats):
    """Map raw stat projections to Yahoo league scoring category values."""
    result = {}
    for cat in cats:
        if cat in proj:
            result[cat] = proj[cat]
        elif cat == "FG%":
            result[cat] = proj["FGM"] / proj["FGA"] if proj.get("FGA", 0) > 0 else 0.0
        elif cat == "FT%":
            result[cat] = proj["FTM"] / proj["FTA"] if proj.get("FTA", 0) > 0 else 0.0
        elif cat == "3PTM":
            result[cat] = proj.get("FG3M", 0.0)
        elif cat == "ST":
            result[cat] = proj.get("STL", 0.0)
        elif cat == "TO":
            result[cat] = proj.get("TOV", 0.0)
        elif cat == "FGM/A":
            result[cat] = proj.get("FGM", 0.0)
        elif cat == "FTM/A":
            result[cat] = proj.get("FTM", 0.0)
    return result

my_cats = to_league_cats(my_proj, categories)
opp_cats = to_league_cats(opp_proj, categories)
classification = classify_categories(my_cats, opp_cats)

df = pd.DataFrame({
    "My Team": my_cats,
    "Opponent": opp_cats,
    "Status": classification,
}).round(3)
print(df.to_string())

       My Team  Opponent  Status
FGM/A  294.570   306.305  tossup
FG%      0.497     0.483  tossup
FTM/A  118.550   131.882    loss
FT%      0.763     0.802    loss
3PTM    70.130    99.227    loss
PTS    777.190   843.592    loss
REB    267.610   287.062    loss
AST    192.020   206.892    loss
ST      48.190    44.700     win
BLK     25.660    29.900    loss
TO      88.170   104.883    loss


In [6]:
# Players on a back-to-back this week
b2b_players = [
    p['name'] for p in my_players
    if p.get('games_detail', {}).get('b2b_count', 0) > 0
]
if b2b_players:
    print(f'⚠️  B2B players: {b2b_players}')

prompt = build_matchup_prompt(my_cats, opp_cats, classification, b2b_players=b2b_players)
advice = ask_gemini(prompt)
print('\n=== WEEKLY GAME PLAN ===\n')
print(advice)


In [7]:
print(prompt)

You are an expert fantasy basketball analyst. Here is this week's H2H category matchup projection:

Category    My Team   Opponent     Status
------------------------------------------
FGM/A         294.6      306.3     tossup
FG%             0.5        0.5     tossup
FTM/A         118.5      131.9       loss
FT%             0.8        0.8       loss
3PTM           70.1       99.2       loss
PTS           777.2      843.6       loss
REB           267.6      287.1       loss
AST           192.0      206.9       loss
ST             48.2       44.7        win
BLK            25.7       29.9       loss
TO             88.2      104.9       loss

Provide a concise weekly strategy: which categories to focus on winning, which to punt, and 2-3 specific streaming/lineup actions to swing toss-up categories.


In [8]:

print("=== MY TEAM ===")
for p in my_players:
  proj = p.get("projected", {}) if "projected" in p else {}
  if not proj and p["stats"]:
      from fantasy.analysis import project_player
      proj = project_player(p["stats"], p["games_remaining"], p.get("status", ""))
  print(f"{p['name']:<25} {p['games_remaining']} games  "
        f"PTS:{proj.get('PTS',0):.1f}  REB:{proj.get('REB',0):.1f}  "
        f"AST:{proj.get('AST',0):.1f}  3PM:{proj.get('FG3M',0):.1f}  "
        f"STL:{proj.get('STL',0):.1f}  BLK:{proj.get('BLK',0):.1f}  "
        f"TOV:{proj.get('TOV',0):.1f}")

print("\n=== OPPONENT ===")
for p in opp_players:
  proj = p.get("projected", {}) if "projected" in p else {}
  if not proj and p["stats"]:
      from fantasy.analysis import project_player
      proj = project_player(p["stats"], p["games_remaining"], p.get("status", ""))
  print(f"{p['name']:<25} {p['games_remaining']} games  "
        f"PTS:{proj.get('PTS',0):.1f}  REB:{proj.get('REB',0):.1f}  "
        f"AST:{proj.get('AST',0):.1f}  3PM:{proj.get('FG3M',0):.1f}  "
        f"STL:{proj.get('STL',0):.1f}  BLK:{proj.get('BLK',0):.1f}  "
        f"TOV:{proj.get('TOV',0):.1f}")

=== MY TEAM ===
Jalen Brunson             3 games  PTS:71.6  REB:10.6  AST:23.4  3PM:7.1  STL:2.6  BLK:0.5  TOV:7.5
Dyson Daniels             4 games  PTS:48.0  REB:28.1  AST:22.2  3PM:1.4  STL:9.0  BLK:1.6  TOV:5.2
Jrue Holiday              4 games  PTS:65.2  REB:17.4  AST:24.2  3PM:9.9  STL:3.8  BLK:0.4  TOV:10.9
Jalen Green               2 games  PTS:38.7  REB:7.7  AST:5.7  3PM:4.7  STL:2.3  BLK:0.3  TOV:5.0
Naji Marshall             3 games  PTS:47.4  REB:13.3  AST:12.0  3PM:2.7  STL:2.6  BLK:0.3  TOV:7.2
Kevin Durant              4 games  PTS:103.6  REB:22.6  AST:16.8  3PM:9.3  STL:3.5  BLK:3.3  TOV:12.5
Evan Mobley               3 games  PTS:58.5  REB:28.0  AST:7.9  3PM:2.9  STL:1.8  BLK:5.3  TOV:4.2
Moussa Diabaté            4 games  PTS:30.0  REB:36.2  AST:11.0  3PM:0.0  STL:3.9  BLK:4.6  TOV:4.8
Tre Jones                 4 games  PTS:56.1  REB:11.9  AST:18.5  3PM:2.7  STL:4.9  BLK:0.5  TOV:5.9
Jayson Tatum              3 games  PTS:57.3  REB:26.7  AST:9.9  3PM:8.4  STL:3.3  BL